# CohortX — Exploration
Key findings from NXML inspection and failure analysis.

In [ ]:
import random
import pandas as pd
from pathlib import Path
from parser import NXMLParser

DATA_DIR = Path.home() / 'Downloads' / 'cohort-x-task-1'
NXML_DIR = DATA_DIR / 'PMC_NXML_Archives'

train_df = pd.read_excel(DATA_DIR / 'Task_1.xlsx', sheet_name='Train')
parser   = NXMLParser()

parsed_map = {}
for pid in train_df['pmcids'].astype(str).str.strip():
    path = NXML_DIR / f'PMC{pid}.nxml'
    if path.exists():
        parsed_map[pid] = parser.parse(str(path))

print(f'Parsed: {len(parsed_map)} / {len(train_df)}')

## 1. NXML Sample Inspection

In [ ]:
sample_pids = random.sample(list(parsed_map.keys()), 5)

for pid in sample_pids:
    parsed = parsed_map[pid]
    gold   = train_df[train_df['pmcids'].astype(str).str.strip() == pid].iloc[0]

    print('='*80)
    print(f'PMC{pid}')
    print(f'\nTITLE:\n  {parsed["title"]}')
    print(f'\nABSTRACT:\n  {parsed["abstract"][:400]}...')
    print(f'\nSECTIONS ({len(parsed["sections"])}):')
    for sec in parsed['sections']:
        print(f'  [{sec["title"][:50]}] {sec["text"][:100]}...')
    print(f'\nKEYWORDS:  {parsed["keywords"][:5]}')
    print(f'\n--- GOLD LABELS ---')
    print(f'  conditions:           {gold["conditions"]}')
    print(f'  study_type:           {gold["study_type"]}')
    print(f'  sex:                  {gold["sex"]}')
    print(f'  minimum_age:          {gold["minimum_age"]}')
    print(f'  maximum_age:          {gold["maximum_age"]}')
    print(f'  eligibility_criteria: {str(gold["eligibility_criteria"])[:300]}')
    print()

## 2. Section Relevance Scoring (MiniLM)
Inspect how well MiniLM ranks sections by relevance to the extraction query.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from config import ELIGIBILITY_QUERY

ranker    = SentenceTransformer('all-MiniLM-L6-v2')
query_emb = ranker.encode(ELIGIBILITY_QUERY, convert_to_tensor=True)

pid      = random.choice(list(parsed_map.keys()))
parsed   = parsed_map[pid]
sections = parsed.get('sections', [])

print(f'PMC{pid} — {parsed["title"][:80]}')
print(f'Total sections: {len(sections)}\n')

scored = []
for sec in sections:
    title_emb = ranker.encode(sec['title'], convert_to_tensor=True)
    text_emb  = ranker.encode(sec['text'][:200], convert_to_tensor=True)
    score     = max(
        float(util.cos_sim(query_emb, title_emb).item()),
        float(util.cos_sim(query_emb, text_emb).item()),
    )
    scored.append((score, sec['title']))

for score, title in sorted(scored, reverse=True):
    print(f'{score:.3f}  [{title[:60]}]')

## 3. Minimum Age Failure Analysis
344/416 documents return 'Not Specified' for minimum_age.  
Root cause: the age information is often absent from the article text entirely — it comes from the ClinicalTrials.gov registration, not the paper.

In [ ]:
# Load a set of predictions to analyse
import re

PREDS_PATH = DATA_DIR / 'submission_ollama.xlsx'   # change to your preds file

if PREDS_PATH.exists():
    preds_df = pd.read_excel(PREDS_PATH)
    gold_df  = train_df[['pmcids','minimum_age']].copy()
    preds_df['pmcids'] = preds_df['pmcids'].astype(str).str.strip()
    gold_df['pmcids']  = gold_df['pmcids'].astype(str).str.strip()

    merged = preds_df.merge(gold_df, on='pmcids', suffixes=('_pred', '_gold'))

    def age_in_text(pid):
        parsed = parsed_map.get(str(pid), {})
        full   = parsed.get('abstract', '') + ' '.join(
            s['text'] for s in parsed.get('sections', [])
        )
        return bool(re.search(r'\b1[5-9]\b|\b[2-9]\d\b', full))

    failures = merged[merged['minimum_age_pred'].isin(
        ['Not Specified', 'None', None, float('nan')]
    )].copy()
    failures['age_in_text'] = failures['pmcids'].apply(age_in_text)

    print(f'Total failures: {len(failures)} / {len(merged)}')
    print('\nAge mention in article text:')
    print(failures['age_in_text'].value_counts())
    print('\nGold value distribution:')
    print(merged['minimum_age_gold'].value_counts().head(10))
else:
    print(f'No predictions file found at {PREDS_PATH}')
    print('Run predict_ollama.py or predict_openai.py first.')

In [ ]:
# Inspect a specific failure case
failure_pids = failures['pmcids'].astype(str).tolist() if PREDS_PATH.exists() else []
print(failure_pids[:10])  # pick one from this list

# Then:
# pid    = '7092565'
# parsed = parsed_map.get(pid, {})
# print(parsed.get('abstract', '')[:600])
# for sec in parsed.get('sections', []):
#     print(f'[{sec["title"]}]', sec['text'][:200])